Loading SQL Magic Commands and Connecting to Database:

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 1000
%sql sqlite:///GRT_GTFS/Trip_Updates_Static_Feed.db

Creating Static Feed Tables (if they don't already exist)

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS static_calendar_dates (
    service_id TEXT,
    date TEXT,
    exception_type INT
);

CREATE TABLE IF NOT EXISTS static_routes (
    route_id INT,
    agency_id TEXT,
    route_short_name TEXT,
    route_long_name TEXT,
    route_desc TEXT,
    route_type INT,
    route_url TEXT
);

CREATE TABLE IF NOT EXISTS static_shapes (
    shape_id INT,
    shape_pt_lat REAL,
    shape_pt_lon REAL,
    shape_pt_sequence INT
);

CREATE TABLE IF NOT EXISTS static_stop_times (
    trip_id INT,
    arrival_time TEXT,
    departure_time TEXT,
    stop_id INT,
    stop_sequence INT,
    pickup_type INT,
    drop_off_type INT
);

CREATE TABLE IF NOT EXISTS static_stops (
    stop_id INT,
    stop_code TEXT,
    stop_name TEXT,
    stop_desc TEXT,
    stop_lat REAL,
    stop_lon REAL,
    zone_id TEXT,
    stop_url TEXT,
    location_type INT,
    parent_station TEXT,
    wheelchair_boarding INT,
    platform_code TEXT
);

CREATE TABLE IF NOT EXISTS static_trips (
    route_id INT,
    service_id TEXT,
    trip_id INT,
    trip_headsign TEXT,
    direction_id INT,
    block_id INT,
    shape_id INT,
    wheelchair_accessible INT,
    bikes_allowed INT
);

Importing (or replacing if already imported) Static Feed Data Into Respective Tables:

In [ ]:
!sqlite3 GRT_GTFS/Trip_Updates_Static_Feed.db "DELETE FROM static_calendar_dates;" ".mode csv" ".import --skip 1 ./GRT_GTFS/Static_Feed/calendar_dates.txt static_calendar_dates"

!sqlite3 GRT_GTFS/Trip_Updates_Static_Feed.db "DELETE FROM static_routes;" ".mode csv" ".import --skip 1 ./GRT_GTFS/Static_Feed/routes.txt static_routes"

!sqlite3 GRT_GTFS/Trip_Updates_Static_Feed.db "DELETE FROM static_shapes;" ".mode csv" ".import --skip 1 ./GRT_GTFS/Static_Feed/shapes.txt static_shapes"

!sqlite3 GRT_GTFS/Trip_Updates_Static_Feed.db "DELETE FROM static_stop_times;" ".mode csv" ".import --skip 1 ./GRT_GTFS/Static_Feed/stop_times.txt static_stop_times"

!sqlite3 GRT_GTFS/Trip_Updates_Static_Feed.db "DELETE FROM static_stops;" ".mode csv" ".import --skip 1 ./GRT_GTFS/Static_Feed/stops.txt static_stops"

!sqlite3 GRT_GTFS/Trip_Updates_Static_Feed.db "DELETE FROM static_trips;" ".mode csv" ".import --skip 1 ./GRT_GTFS/Static_Feed/trips.txt static_trips"

Selecting Static Feed Provided Data:

In [ ]:
%%sql

SELECT service_id, date
FROM static_calendar_dates
LIMIT 5;

In [ ]:
%%sql

SELECT route_id, route_long_name
FROM static_routes
ORDER BY route_id;

In [ ]:
%%sql

SELECT *
FROM static_shapes
LIMIT 5;

In [ ]:
%%sql

SELECT trip_id, arrival_time, departure_time, stop_id, stop_sequence
FROM static_stop_times
LIMIT 5;

In [ ]:
%%sql

SELECT stop_id, stop_code, stop_name, stop_lat, stop_lon
FROM static_stops
LIMIT 10;

In [ ]:
%%sql

SELECT route_id, service_id, trip_id, trip_headsign, direction_id, block_id, shape_id
FROM static_trips
LIMIT 5;

Selecting Trip Updates Provided Data (which were already imported with Trip_Updates_Decoder.py into respective tables):

In [ ]:
%%sql

SELECT fetched_at, timestamp
FROM feed_header_update 
LIMIT 5;

In [ ]:
%%sql

SELECT stop_time_update_id, fetched_at, trip_entity_id, stop_sequence, stop_id, arrival_time, departure_time
FROM stop_time_update
LIMIT 5;

In [ ]:
%%sql

SELECT trip_update_id, fetched_at, entity_id, timestamp, trip_id, route_id, start_time, start_date
FROM trip_update
LIMIT 5;

Analysis Question 1: How well does GRT run on schedule, and which routes are least reliable?

In [ ]:
%%sql


CREATE INDEX IF NOT EXISTS static_trips_trip_id_idx ON static_trips (trip_id);
CREATE INDEX IF NOT EXISTS static_trips_route_id_idx ON static_trips (route_id);

CREATE INDEX IF NOT EXISTS static_stop_times_trip_id_idx ON static_stop_times (trip_id);

CREATE INDEX IF NOT EXISTS static_routes_route_id_idx ON static_routes (route_id);

CREATE INDEX IF NOT EXISTS trip_update_trip_id_idx ON trip_update (trip_id);
CREATE INDEX IF NOT EXISTS trip_update_route_id_idx ON trip_update (route_id);
CREATE INDEX IF NOT EXISTS trip_update_entity_fetched_idx ON trip_update (entity_id, fetched_at);

CREATE INDEX IF NOT EXISTS stop_time_update_entity_fetched_idx ON stop_time_update (trip_entity_id, fetched_at);
CREATE INDEX IF NOT EXISTS stop_time_update_entity_stopseq_fetched_idx ON stop_time_update (trip_entity_id, stop_sequence, fetched_at);

In [ ]:
%%sql

SELECT static_trips.trip_id, static_trips.route_id, 
    static_stop_times.stop_sequence, static_stop_times.stop_id, static_stop_times.arrival_time, static_stop_times.departure_time,
    trip_update.start_time, MAX(trip_update.fetched_at) AS latest_fetched_at, trip_update.start_date,
    SUBSTR(datetime(stop_time_update.arrival_time, 'unixepoch', '-4 hours'), 12, 8) AS actual_arrival_time, SUBSTR(datetime(stop_time_update.departure_time, 'unixepoch', '-4 hours'), 12, 8) AS actual_departure_time,
    
    ROUND(((stop_time_update.arrival_time - 4 * 3600) - (
        strftime('%s', substr(trip_update.start_date, 1, 4)||'-'||substr(trip_update.start_date, 5, 2)||'-'||substr(trip_update.start_date, 7, 2))
        + CAST(substr(static_stop_times.arrival_time, 1, 2) AS INT) * 3600
        + CAST(substr(static_stop_times.arrival_time, 4, 2) AS INT) * 60
        + CAST(substr(static_stop_times.arrival_time, 7, 2) AS INT) )) / 60.0, 1) AS arrival_difference,

    ROUND(((stop_time_update.departure_time - 4 * 3600) - (
        strftime('%s', substr(trip_update.start_date, 1, 4)||'-'||substr(trip_update.start_date,5,2)||'-'||substr(trip_update.start_date, 7, 2))
        + CAST(substr(static_stop_times.departure_time, 1, 2) AS INT) * 3600
        + CAST(substr(static_stop_times.departure_time, 4, 2) AS INT) * 60
        + CAST(substr(static_stop_times.departure_time, 7, 2) AS INT) )) / 60.0, 1) AS departure_difference

FROM static_trips

LEFT JOIN static_stop_times
ON static_trips.trip_id = static_stop_times.trip_id

JOIN trip_update
ON trip_update.trip_id = static_trips.trip_id

JOIN stop_time_update
ON stop_time_update.trip_entity_id = trip_update.entity_id
    AND stop_time_update.fetched_at = trip_update.fetched_at
    AND stop_time_update.stop_sequence = static_stop_times.stop_sequence

WHERE static_trips.trip_id = 4164599

GROUP BY trip_update.trip_id, stop_time_update.stop_sequence, trip_update.start_date
ORDER BY trip_update.fetched_at

LIMIT 500;

In [ ]:
%%sql

SELECT static_trips.trip_id, static_trips.route_id,
    ROUND(AVG(((stop_time_update.departure_time - 4 * 3600) - (
        strftime('%s', substr(trip_update.start_date, 1, 4)||'-'||substr(trip_update.start_date, 5, 2)||'-'||substr(trip_update.start_date, 7, 2))
        + CAST(substr(static_stop_times.departure_time, 1, 2) AS INT) * 3600
        + CAST(substr(static_stop_times.departure_time, 4, 2) AS INT) * 60
        + CAST(substr(static_stop_times.departure_time, 7, 2) AS INT))) / 60.0), 1) AS average_delay,

        ROUND(MAX(((stop_time_update.departure_time - 4 * 3600) - (
        strftime('%s', substr(trip_update.start_date, 1, 4)||'-'||substr(trip_update.start_date, 5, 2)||'-'||substr(trip_update.start_date, 7, 2))
        + CAST(substr(static_stop_times.departure_time, 1, 2) AS INT) * 3600
        + CAST(substr(static_stop_times.departure_time, 4, 2) AS INT) * 60
        + CAST(substr(static_stop_times.departure_time, 7, 2) AS INT))) / 60.0), 1) AS max_delay

FROM static_trips

LEFT JOIN static_stop_times
ON static_trips.trip_id = static_stop_times.trip_id

JOIN trip_update
ON trip_update.trip_id = static_trips.trip_id

JOIN stop_time_update
ON stop_time_update.trip_entity_id = trip_update.entity_id
    AND stop_time_update.fetched_at = trip_update.fetched_at
    AND stop_time_update.stop_sequence = static_stop_times.stop_sequence

WHERE NOT ABS((((stop_time_update.departure_time - 4 * 3600) - (
        strftime('%s', substr(trip_update.start_date, 1, 4)||'-'||substr(trip_update.start_date, 5, 2)||'-'||substr(trip_update.start_date, 7, 2))
        + CAST(substr(static_stop_times.departure_time, 1, 2) AS INT) * 3600
        + CAST(substr(static_stop_times.departure_time, 4, 2) AS INT) * 60
        + CAST(substr(static_stop_times.departure_time, 7, 2) AS INT))) / 60.0)) > 120

GROUP BY trip_update.route_id
ORDER BY trip_update.route_id

LIMIT 1000;

Analysis Question 2: 

Analysis Question 3:

Data Visualizing:

In [ ]:
# matplotlib and seaborn visualization

import matplotlib.pyplot as plt
import seaborn as sns